# EMQA Assignment 2 — starter notebook
**Risk, contracts and forecast-driven decisions for your role** (lectures 4–7).

Read the brief (`A2_brief_EN.pdf`) first. Choose the role closest to your company (B battery, H hydro, R solar, S supplier/industrial consumer). Public data only: do not use, and do not disclose, confidential company data.

In [ ]:
# Google Colab: copy the course repository, so that the data and the saved results are found
import os, subprocess, sys
if 'google.colab' in sys.modules and not os.getcwd().startswith('/content/EMQA'):
    if not os.path.isdir('/content/EMQA'):
        subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://github.com/danpele/EMQA.git', '/content/EMQA'], check=True)
    os.chdir('/content/EMQA/assignments/A2')

In [ ]:
STUDENT_ID = 'TODO-your-student-ID'
ROLE = 'B'          # B, H, R or S
MODE = 'personal'

import os, sys, json, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
sys.path.insert(0, os.path.abspath(".."))      # folder that contains emqa_assign.py
import emqa_assign as ea
pd.set_option("display.width", 160)

p = ea.params(STUDENT_ID, 2, ROLE)
if MODE == "latest":
    p = ea.latest_window(p)
print(json.dumps(p, indent=1))
display(ea.data_version())
P, L, R = ea.panel("RO")
t0, t1 = pd.Timestamp(p["test_start"]), pd.Timestamp(p["test_end"])
s0, s1 = pd.Timestamp(p["selection_start"]), pd.Timestamp(p["selection_end"])
sel = pd.date_range(s0, s1, freq="D")     # (2) SELECTION: choose rule, alpha, w*, K, PPA prices, toll here only
test = pd.date_range(t0, t1, freq="D")    # (3) TEST: choices frozen; report results here only
fdays = pd.date_range(s0 - pd.Timedelta(days=p["burn_in_days"]), t1, freq="D")   # (1) burn-in + selection + test: forecast ALL of these
print("selection:", s0.date(), "->", s1.date(), "| test:", t0.date(), "->", t1.date())

## Task 1 — Exposure profile (normalised public data)
Use exactly the profile definition of your role in the brief (Table 1). Example below for each role.

In [ ]:
if ROLE == "H":
    res = ea.day_matrix("hydro_water_reservoir")[ea.HOURS] * p["fleet_share"]   # MW = MWh per hour
    ror = ea.day_matrix("hydro_run_of_river")[ea.HOURS] * p["fleet_share"]
    Q = (res + ror).reindex(P.index)
elif ROLE == "R":
    s = ea.day_matrix("solar")[ea.HOURS]
    peak_ref = s.max(axis=1).rolling(365, min_periods=180).max().shift(2)       # trailing peak up to d-2 (metered at gate closure)
    Q = (s.div(peak_ref, axis=0).clip(upper=1) * p["pv_mwp"]).reindex(P.index)
elif ROLE == "S":
    l = ea.day_matrix("load")[ea.HOURS]
    ref = l.mean(axis=1).rolling(365, min_periods=180).mean().shift(2)     # trailing mean up to d-2
    Q = (l.div(ref, axis=0) * p["avg_load_mw"]).reindex(P.index)
else:
    Q = None   # battery: no exogenous volume; see Task 3
# TODO: monthly table 2023 -> test end: volume (MWh), cash flow (EUR), base price, capture price, capture rate
# TODO: clean spark spread table for the months of your test window (brief, eq. 1)

## Task 2 — Probabilistic forecasts: LEAR (from Assignment 1) vs one machine-learning model

In [ ]:
F = {}        # "Naive", "LEAR", "<your ML model>"  -> DataFrame (fdays x 24)
Q99 = {}      # "<model> + CP" / "+ HS"            -> array (len(sel) + len(test), 24, 99): selection AND test days
T2 = pd.DataFrame(columns=["MAE", "RMSE", "APS", "cov50", "cov90"])
# TODO   (optional bonus: a foundation model such as Chronos-2 or TabPFN; record its version and pre-training cutoff)

## Task 3 — Decision and decision ledger
Evaluate every signal on the selection period, choose and FREEZE the rule (and alpha) there, then report all signals on the test period. One ledger row per test day. Required columns below.

In [ ]:
LEDGER_COLUMNS = ["date", "info_cutoff", "forecast_used", "decision", "volume_MWh", "settlement_price_summary",
                  "cash_flow_EUR", "baseline_cash_flow_EUR", "difference_EUR"]
ledger = pd.DataFrame(columns=LEDGER_COLUMNS)
# TODO: strategies: oracle (perfect foresight, upper bound only), naive, point forecast, (B and R) quantile rule for each alpha in p["risk_knob"]
def select_fn(P, L, R):
    """Your EX ANTE choices, computed from the SELECTION period only (days in `sel`). Return a dict, e.g.
    {"rule": "LEAR point", "alpha": None, "w_star": ..., "K": ..., "toll": ...}."""
    # TODO
    raise NotImplementedError
# ea.selection_check(select_fn, P, L, R, p)      # must print PASSED

## Task 4 — Energy risk: price-volume decomposition, role risk, imbalance volume, RO-HU basis, CFaR

In [ ]:
# X: daily cash flow of your BASELINE exposure (brief, Table 1), from 2022 to test end (kEUR; battery: EUR)
HU = ea.day_matrix("HU")[ea.HOURS].reindex(P.index)      # Hungarian prices for the RO-HU basis
T4 = {}
# TODO: (a) monthly decomposition vs m-12, corr(volume, base price); (b) role risk; (c) imbalance volume with Q~[d,h] = Q[d-2,h] (last metered day);
#       (d) RO-HU spread statistics and figure; (e) monthly CFaR95 and daily CFaR95 / ES-type number

## Task 5 — VaR and ES of your cash flow, Kupiec and Christoffersen backtests

In [ ]:
T5 = pd.DataFrame(columns=["method", "alpha", "breaches", "expected", "Kupiec_p", "Christoffersen_p",
                           "mean_VaR", "mean_ES", "ES_gap_mean", "ES_gap_CI90"])
# TODO: HS, Normal, Monte Carlo AR(1,7)-GARCH(1,1)-t; window p["var_window"]; alphas p["alphas"]
#       information at gate closure: X only up to day d-2 (two-day-ahead forecast)

## Task 6 — Contract or hedge design

In [ ]:
T6 = pd.DataFrame()
# TODO (role-specific, see brief; H and S also with settlement on the HU base price)
# K, w*, the PPA prices and the toll are fixed on the SELECTION period (select_fn); the table is computed on the TEST period

## Task 7 — Robustness and adverse scenario

In [ ]:
ADVERSE = {}
# TODO

## Numbers for the memo

In [ ]:
MY_NUMBERS = {"params": p}
# TODO: add every table and number quoted in the memo, and the reconciliation of p["check_days"]
ledger.to_csv(f"ledger_{p['student_id']}_{ROLE}_{MODE}.csv", index=False)
with open(f"my_numbers_A2_{p['student_id']}_{ROLE}_{MODE}.json", "w") as fh:
    json.dump(MY_NUMBERS, fh, indent=1, default=str)